# Chatrud Geochemical Anomaly Detection

Stream sediment anomaly detection for an ophiolitic terrain. The workflow combines an Isolation Forest with percentile-based extreme-value filters and deposit-specific pathfinder rules (chromite, VMS, epithermal, porphyry).

The reusable logic lives in the `chatrud_anomaly` package; this notebook walks through it step by step.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd

from chatrud_anomaly import (
    DEPOSIT_MODELS,
    GENERAL_MODEL,
    apply_filter,
    detect_anomalies,
    load_geochemical_data,
    log_scale,
)
from chatrud_anomaly.data import TARGET_ELEMENTS
from chatrud_anomaly.plotting import (
    apply_style,
    enrichment_table,
    plot_correlation_heatmap,
    plot_distribution_with_threshold,
    plot_enrichment,
    plot_multi_model_map,
    plot_radar,
    plot_raw_vs_log,
    plot_reduction_flow,
    plot_score_distribution,
    plot_spatial_map,
    plot_target_validation,
)
from chatrud_anomaly.reporting import HOVER_COLUMNS, write_excel_report

apply_style()

DATA_PATH = Path("../data/Chatrud_Final_data.xlsx")
OUTPUT_PATH = Path("../outputs/Chatrud_Anomalies_Report.xlsx")

## 2. Data Loading and Preparation

The workbook holds laboratory metadata in its first two rows (measurement units and detection limits). Values below the limit of detection (`<LOD`) are replaced with `LOD × 0.75`, so non-detects stay in the dataset without introducing missing values.

In [ ]:
data = load_geochemical_data(DATA_PATH, sheet_name="Sheet2")
print(f"Number of processed samples: {len(data)}")
data[TARGET_ELEMENTS].describe().T

## 3. Exploratory Data Analysis

Spearman correlations are rank-based, so they are identical on raw, log-transformed and scaled data.

In [ ]:
plot_correlation_heatmap(data, TARGET_ELEMENTS)

In [ ]:
for element in ["Cr", "Ni"]:
    plot_distribution_with_threshold(data[element], element)

## 4. Transformation and Scaling

`log1p` reduces the skewness of geochemical distributions, and `RobustScaler` (median and IQR) keeps the outliers we want to detect from distorting the scale.

In [ ]:
log_values, scaled = log_scale(data, TARGET_ELEMENTS)

In [ ]:
plot_raw_vs_log(data, log_values, ["Cr", "Ni", "Co", "Cu", "Zn", "Pb"])

In [ ]:
plot_raw_vs_log(data, log_values, ["Au", "Ag", "As", "Sb", "Fe", "Mg", "Ti", "Al", "Mo"])

## 5. Anomaly Detection with Isolation Forest

Isolation Forest isolates observations through random splits. `contamination=0.10` flags the 10% most isolated samples; negative decision-function values mark anomalies.

In [ ]:
scored, anomalies = detect_anomalies(data, scaled, n_estimators=500, contamination=0.10, random_state=42)
print(f"Total anomalies detected: {len(anomalies)}")

In [ ]:
plot_score_distribution(scored["score_value"])

## 6. Spatial Distribution of Anomalies

In [ ]:
hover = ["Name"] + TARGET_ELEMENTS
plot_spatial_map(scored, "Spatial Distribution of Geochemical Anomalies (Chatrud)", hover)

## 7. Percentile Filter

An Isolation Forest flags statistical outlyingness in any direction. To keep only enriched samples, a candidate must reach the 95th percentile (computed on the full dataset) in at least one key pathfinder element.

In [ ]:
general_targets = apply_filter(anomalies, scored, GENERAL_MODEL)
print(f"Isolation Forest candidates: {len(anomalies)}")
print(f"Passing the percentile filter: {len(general_targets)}")
general_targets[["Name", "Cr", "Ni", "Cu", "Au"]].head()

In [ ]:
plot_spatial_map(general_targets, "Spatial Distribution of Refined Geochemical Anomalies (Chatrud)", hover)

### 7.1 Anomalies vs Background

Mean concentration of the refined anomalies against the full dataset, expressed as an enrichment factor.

In [ ]:
enrichment = enrichment_table(general_targets, scored, TARGET_ELEMENTS)
plot_enrichment(enrichment, "Refined Anomaly Signature (IF + Percentile Filter)")
enrichment["Enrichment Factor"]

## 8. Deposit-Type Models

In [ ]:
for model in DEPOSIT_MODELS:
    print(f"{model.label}: {model.pathfinders}, quantile={model.quantile}, min elements={model.min_elements}")

In [ ]:
deposit_targets = {model.key: apply_filter(anomalies, scored, model) for model in DEPOSIT_MODELS}
{key: len(frame) for key, frame in deposit_targets.items()}

In [ ]:
for model in DEPOSIT_MODELS:
    targets = deposit_targets[model.key]
    if targets.empty:
        continue
    display(plot_radar(scaled, targets.index, list(model.pathfinders), title=f"{model.label}: Geochemical Fingerprint"))
    for x, y in model.validation_pairs:
        display(plot_target_validation(scored, anomalies, targets, x, y, model.label, annotate=True))

In [ ]:
targets_by_label = {m.label: deposit_targets[m.key] for m in DEPOSIT_MODELS if not deposit_targets[m.key].empty}
plot_multi_model_map(targets_by_label, HOVER_COLUMNS)

## 9. Sample Reduction Summary

In [ ]:
counts = {
    "Raw Data": len(scored),
    "Isolation Forest": len(anomalies),
    "Percentile Filter": len(general_targets),
}
plot_reduction_flow(counts)

## 10. Export

In [ ]:
from chatrud_anomaly.pipeline import PipelineResult

result = PipelineResult(
    elements=TARGET_ELEMENTS,
    scored=scored,
    log_values=log_values,
    scaled=scaled,
    anomalies=anomalies,
    general_targets=general_targets,
    deposit_targets=deposit_targets,
)
print(f"Report written to {write_excel_report(result, OUTPUT_PATH)}")